In [1]:
import pandas as pd
import igraph
import pickle as pk
import os

In [4]:
def describe(g):
    print( g.vcount() )
    print( g.ecount() )
    print( g.vertex_attributes() )
    print( g.edge_attributes() )

In [3]:
os.listdir("graphs/")

['humanities_graph.pk', 'humanities_cocitation_graph.pk']

In [14]:
dataset = 'Humanities'

# Create cocitation graph

In [2]:
## Load graph
g = igraph.Graph.Read_Pickle( 'graphs/' + dataset + '_graph.pk' )
describe(g)

In [6]:
cocitation_matrix = g.cocitation()

In [7]:
gc = igraph.Graph.Weighted_Adjacency(
    cocitation_matrix,
    mode='undirected',
    attr='weight',
    loops=False
)

for attr in g.vertex_attributes():
    gc.vs[attr] = g.vs[attr]

In [8]:
describe(gc)

34270
6476008
['name', 'title', 'school', 'type', 'authors']
['weight']


In [9]:
## Write to pickle
gc.write_pickle( "graphs/" + dataset + "_cocitation_graph.pk")

# Visualize

In [10]:
from fa2_modified import ForceAtlas2
import random

In [17]:
gc = igraph.Graph.Read_Pickle( 'graphs/' + dataset + '_cocitation_graph.pk' )

### Filter and layout

In [20]:
filt_thresh = 10

In [18]:
# Delete edges with cocitation weight less than threshold
gc.es.select(weight_lt = filt_thresh).delete()

# Delete vertices with zero degree
gc.vs.select(_degree_lt=1).delete()

print( gc.vcount() )
print( gc.ecount() )

27735
69363


In [19]:
random.seed(1327)

forceatlas2 = ForceAtlas2(
                        # Behavior alternatives
                        outboundAttractionDistribution=False,  # Dissuade hubs
                        linLogMode=False,  # NOT IMPLEMENTED
                        adjustSizes=False,  # Prevent overlap (NOT IMPLEMENTED)
                        edgeWeightInfluence=1.5,

                        # Performance
                        jitterTolerance=1.0,  # Tolerance
                        barnesHutOptimize=True,
                        barnesHutTheta=1.2,
                        multiThreaded=False,  # NOT IMPLEMENTED

                        # Tuning
                        scalingRatio=1.0,
                        strongGravityMode=True,
                        gravity=2,

                        # Log
                        verbose=True)

l = forceatlas2.forceatlas2_igraph_layout( gc, pos=None, iterations=100, weight_attr='weight' )

100%|████████████████████████████████████████████████████████████| 100/100 [06:22<00:00,  3.83s/it]

BarnesHut Approximation  took  328.86  seconds
Repulsion forces  took  52.81  seconds
Gravitational forces  took  0.01  seconds
Attraction forces  took  0.22  seconds
AdjustSpeedAndApplyForces step  took  0.33  seconds


In [25]:
# Write layout
with open("layouts/" + dataset + "_" + str(filt_thresh) +  ".pkl", "wb") as f:
    pk.dump(l, f)

### Plot

In [29]:
with open("layouts/" + dataset + "_" + str(filt_thresh) + ".pkl", "rb") as f:
    l = pk.load(f)

In [ ]:
igraph.plot(
    gc,
    'plots/' + dataset + "_" + str(filt_thresh) + '_cocitations.png', 
    layout=l,
    vertex_size = 1,
    #vertex_size = [(s/200)+2 for s in gc.vs['strength']],
    edge_width=0.1
)